In [52]:
from pystac_client import Client

# Connecting to Sentinel-2
# Copernicus Data Space STAC catalog
STAC_URL = "https://stac.dataspace.copernicus.eu/v1"

catalog = Client.open(STAC_URL)

print("Connected to Copernicus!")

Connected to Copernicus!


In [53]:
# %%
# Load the plantation GeoJSON

import geopandas as gpd

plantation_gdf = gpd.read_file("../data/boundaries/PT.geojson")

print("Number of features:", len(plantation_gdf))
print(plantation_gdf[["field_3", "begin", "end", "Acreage"]])

Number of features: 2
             field_3  begin   end  Acreage
0  PT. PALMINA UTAMA   2216  2338  3711923
1  PT. PALMINA UTAMA   2216  2338  3711923


In [54]:
# %%
# Inspect plantation geometries

for i, feature in plantation_gdf.iterrows():
    print(f"\nFeature {i}")
    print("Plantation:", feature["field_3"])
    print("Geometry type:", feature.geometry.geom_type)
    print("Bounds:", feature.geometry.bounds)


Feature 0
Plantation: PT. PALMINA UTAMA
Geometry type: Polygon
Bounds: (111.5023, -0.1126, 111.5245, -0.08915)

Feature 1
Plantation: PT. PALMINA UTAMA
Geometry type: Polygon
Bounds: (111.3959, -0.04155, 111.4481, 0.031176)


In [55]:
# %%
# Select the first plantation block

plantation = plantation_gdf.iloc[0]

print("Plantation:", plantation["field_3"])
print("Area:", plantation["Acreage"])
print("Geometry:", plantation.geometry.geom_type)

Plantation: PT. PALMINA UTAMA
Area: 3711923
Geometry: Polygon


In [56]:
# %%
# Convert the plantation polygon to GeoJSON

plantation_geometry = plantation.geometry.__geo_interface__

print(plantation_geometry["type"])

Polygon


In [57]:
# %%
# Get bounding box from the actual plantation polygon

min_x, min_y, max_x, max_y = plantation.geometry.bounds

bbox = [
    min_x,
    min_y,
    max_x,
    max_y
]

print("Bounding box:")
print(bbox)

Bounding box:
[111.5023, -0.1126, 111.5245, -0.08915]


In [58]:
# %%
# Search Sentinel-2 images over the actual plantation

search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=plantation_geometry,
    datetime="2026-01-01/2026-09-09",
    query={
        "eo:cloud_cover": {
            "lte": 20
        }
    },
    max_items=100
)

items = list(search.items())

print(f"Found {len(items)} Sentinel-2 images")

Found 2 Sentinel-2 images


In [59]:
# %%
# Examine available Sentinel-2 images
# Gets available images based on the coors above

for item in items:
    print(
        item.datetime.date(),
        round(item.properties.get("eo:cloud_cover", 999), 2),
        item.id
    )

2026-09-02 18.87 S2C_MSIL2A_20260902T024521_N0512_R132_T49MEV_20260902T073417
2026-08-08 1.13 S2B_MSIL2A_20260808T024529_N0512_R132_T49MEV_20260808T045442


In [60]:
# %%
# Check the plantation bounding box

print("min_x:", min_x)
print("min_y:", min_y)
print("max_x:", max_x)
print("max_y:", max_y)

print("\nBBOX:")
print(bbox)

min_x: 111.5023
min_y: -0.1126
max_x: 111.5245
max_y: -0.08915

BBOX:
[111.5023, -0.1126, 111.5245, -0.08915]


In [61]:
# %%
# Load Sentinel Hub credentials

import os
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("SENTINELHUB_CLIENT_ID")
client_secret = os.getenv("SENTINELHUB_CLIENT_SECRET")

print("Client ID loaded:", client_id is not None)
print("Client Secret loaded:", client_secret is not None)

Client ID loaded: True
Client Secret loaded: True


In [62]:
# %%
# Authenticate with Copernicus

import requests

token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

response = requests.post(
    token_url,
    data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    },
)

print("Status code:", response.status_code)

if response.ok:
    token = response.json()["access_token"]
    print("Authentication successful!")
else:
    print("Authentication failed:")
    print(response.text)

Status code: 200
Authentication successful!


In [63]:
# %%
# Reproject plantation to UTM Zone 49S
# UTM Zone 49S = EPSG:32749

plantation_utm = plantation_gdf.to_crs("EPSG:32749")

print("Original CRS:")
print(plantation_gdf.crs)

print("\nNew CRS:")
print(plantation_utm.crs)

Original CRS:
EPSG:4326

New CRS:
EPSG:32749


In [64]:
# %%
# Select the first plantation polygon in UTM coordinates

plantation_utm_feature = plantation_utm.iloc[0]

min_x_utm, min_y_utm, max_x_utm, max_y_utm = (
    plantation_utm_feature.geometry.bounds
)

bbox_utm = [
    min_x_utm,
    min_y_utm,
    max_x_utm,
    max_y_utm
]

print("UTM bounding box:")
print(bbox_utm)

print("\nWidth in meters:")
print(max_x_utm - min_x_utm)

print("\nHeight in meters:")
print(max_y_utm - min_y_utm)

UTM bounding box:
[555894.0354921706, 9987553.80792723, 558364.4519784447, 9990145.865082655]

Width in meters:
2470.4164862741018

Height in meters:
2592.057155424729


In [65]:
# %%
# Request Sentinel-2 Red (B04) and NIR (B08)
# August 8, 2026
# 10-meter resolution

import requests

evalscript = """
//VERSION=3

function setup() {
    return {
        input: ["B04", "B08"],
        output: {
            bands: 2,
            sampleType: "FLOAT32"
        }
    };
}

function evaluatePixel(sample) {
    return [sample.B04, sample.B08];
}
"""

request_body = {
    "input": {
        "bounds": {
            "bbox": bbox_utm,
            "properties": {
                "crs": "http://www.opengis.net/def/crs/EPSG/0/32749"
            }
        },
        "data": [
            {
                "type": "sentinel-2-l2a",
                "dataFilter": {
                    "timeRange": {
                        "from": "2026-08-08T00:00:00Z",
                        "to": "2026-08-09T00:00:00Z"
                    }
                }
            }
        ]
    },
    "output": {
        "resx": 10,
        "resy": 10,
        "responses": [
            {
                "identifier": "default",
                "format": {
                    "type": "image/tiff"
                }
            }
        ]
    },
    "evalscript": evalscript
}

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(
    "https://sh.dataspace.copernicus.eu/process/v1",
    headers=headers,
    json=request_body
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))
print("Response size:", len(response.content), "bytes")

if response.ok:
    print("Sentinel-2 imagery successfully retrieved!")
else:
    print("Request failed:")
    print(response.text)

Status code: 200
Content type: image/tiff
Response size: 318274 bytes
Sentinel-2 imagery successfully retrieved!


In [66]:
# %%
# Load the Sentinel-2 GeoTIFF from the API response

import rasterio
from rasterio.io import MemoryFile

with MemoryFile(response.content) as memfile:
    with memfile.open() as dataset:

        print("Number of bands:", dataset.count)
        print("Width:", dataset.width, "pixels")
        print("Height:", dataset.height, "pixels")
        print("CRS:", dataset.crs)
        print("Data type:", dataset.dtypes)

        # Band 1 = B04 Red
        # Band 2 = B08 NIR
        red = dataset.read(1)
        nir = dataset.read(2)

print("\nRed band shape:", red.shape)
print("NIR band shape:", nir.shape)

Number of bands: 2
Width: 247 pixels
Height: 259 pixels
CRS: EPSG:32749
Data type: ('float32', 'float32')

Red band shape: (259, 247)
NIR band shape: (259, 247)


In [67]:
# %%
# Calculate NDVI

import numpy as np

# NDVI = (NIR - Red) / (NIR + Red)

denominator = nir + red

ndvi = np.where(
    denominator != 0,
    (nir - red) / denominator,
    np.nan
)

print("NDVI calculated successfully!")

print("\nNDVI statistics:")
print("Minimum:", np.nanmin(ndvi))
print("Maximum:", np.nanmax(ndvi))
print("Mean:", np.nanmean(ndvi))
print("Median:", np.nanmedian(ndvi))
print("Standard deviation:", np.nanstd(ndvi))

NDVI calculated successfully!

NDVI statistics:
Minimum: 0.099635735
Maximum: 0.8778523
Mean: 0.69830775
Median: 0.75330687
Standard deviation: 0.14729308
